In [1]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    brier_score_loss
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
train_df = pd.read_json("/home/tawe7542/Roberta_base/train.jsonl", lines=True)
val_df   = pd.read_json("/home/tawe7542/Roberta_base/val.jsonl", lines=True)

In [3]:
print(train_df.head())
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

                                     id  \
0  ea468d03-1973-5039-86b2-ff225bb92c4e   
1  0d05f269-6d67-521d-9b5d-cc18f482c6c1   
2  c2ec79f3-da80-58f8-bef0-3e0ea7ab072f   
3  4ad37c58-0bb7-536b-997d-cfccabd0d094   
4  07747b0c-5051-5e0d-8096-b4d4ed8bd98e   

                                                text  \
0  Duke Ellington, a titan of jazz, revolutionize...   
1  I reflected on the shifting dynamics of media ...   
2  In F. Scott Fitzgerald's "The Great Gatsby," t...   
3  I still chuckle when I think about that time I...   
4  Yoga, originating in ancient India, encompasse...   

                          model  label   genre  
0          falcon3-10b-instruct      1  essays  
1                       o3-mini      1  essays  
2                        gpt-4o      1  essays  
3  deepseek-r1-distill-qwen-32b      1  essays  
4              gemini-2.0-flash      1  essays  
label
1    14606
0     9101
Name: count, dtype: int64
label
1    2312
0    1277
Name: count, dtype: int64


In [4]:
#Adding stylometric features
def extract_features(text):
    words = text.split()
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    
    num_words = len(words)
    num_chars = len(text)
    num_sentences = max(len(sentences), 1)

    avg_word_len = np.mean([len(w) for w in words]) if words else 0.0
    avg_sentence_len = num_words / num_sentences

    punctuation_count = len(re.findall(r"[.,!?;:]", text))
    uppercase_ratio = sum(1 for c in text if c.isupper()) / max(num_chars, 1)
    digit_ratio = sum(1 for c in text if c.isdigit()) / max(num_chars, 1)
    space_ratio = sum(1 for c in text if c.isspace()) / max(num_chars, 1)

    unique_words = len(set(w.lower() for w in words)) if words else 0
    lexical_diversity = unique_words / max(num_words, 1)

    return [
        num_words,
        num_chars,
        avg_word_len,
        avg_sentence_len,
        punctuation_count,
        uppercase_ratio,
        digit_ratio,
        space_ratio,
        lexical_diversity,
    ]

X_train_feat = np.array([extract_features(t) for t in train_df["text"]], dtype=np.float32)
X_val_feat   = np.array([extract_features(t) for t in val_df["text"]], dtype=np.float32)

In [5]:
#Normalize features
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_val_feat   = scaler.transform(X_val_feat)

In [6]:
#Tokenization
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    max_length=512
)

val_encodings = tokenizer(
    val_df["text"].tolist(),
    truncation=True,
    max_length=512
)

In [7]:
#Dataset
class FusionDataset(Dataset):
    def __init__(self, encodings, features, labels):
        self.encodings = encodings
        self.features = features
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: self.encodings[k][idx] for k in self.encodings}
        item["features"] = torch.tensor(self.features[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = FusionDataset(train_encodings, X_train_feat, train_df["label"].values)
val_dataset   = FusionDataset(val_encodings, X_val_feat, val_df["label"].values)

In [8]:
#Data collator
class FusionDataCollator:
    def __init__(self, tokenizer):
        self.base_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        text_features = []
        extra_features = []
        labels = []

        for f in features:
            text_features.append({
                "input_ids": f["input_ids"],
                "attention_mask": f["attention_mask"]
            })
            extra_features.append(f["features"])
            labels.append(f["labels"])

        batch = self.base_collator(text_features)
        batch["features"] = torch.stack(extra_features)
        batch["labels"] = torch.stack(labels)
        return batch

data_collator = FusionDataCollator(tokenizer)


In [9]:
#Fusion model
class FusionModel(nn.Module):
    def __init__(self, model_name, num_features, num_labels=2, dropout=0.3):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.text_dropout = nn.Dropout(dropout)

        self.feature_net = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 32, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_labels)
        )
    
    def forward(self, input_ids, attention_mask, features, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS-like  token representation
        text_repr = outputs.last_hidden_state[:, 0, :]
        text_repr = self.text_dropout(text_repr)

        feat_repr = self.feature_net(features)

        combined = torch.cat([text_repr, feat_repr], dim=1)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = FusionModel(
    model_name=model_name,
    num_features=X_train_feat.shape[1],
    num_labels=2,
    dropout=0.3
)


Loading weights: 100%|██████████| 197/197 [00:01<00:00, 178.64it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# Evaluation Metrics
def compute_pan_metrics(eval_pred):
    logits, y_true = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    y_pred = (probs >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_true, probs)
    brier = 1 - brier_score_loss(y_true, probs)

    n = len(y_true)
    non_answers = np.sum(probs == 0.5)
    correct = np.sum(y_pred == y_true)

    if n - non_answers == 0:
        c_at_1 = 0.0
    else:
        c_at_1 = (correct + non_answers * (correct / (n - non_answers))) / n

    f1 = f1_score(y_true, y_pred, zero_division=0)

    beta = 0.5
    adjusted_preds = y_pred.copy()
    adjusted_preds[probs == 0.5] = 0

    precision = precision_score(y_true, adjusted_preds, zero_division=0)
    recall = recall_score(y_true, adjusted_preds, zero_division=0)

    if precision + recall == 0:
        f05u = 0.0
    else:
        f05u = (1 + beta**2) * precision * recall / ((beta**2 * precision) + recall)

    cm = confusion_matrix(y_true, y_pred)
    mean_score = np.mean([roc_auc, brier, c_at_1, f1, f05u])

    return {
        "roc_auc": roc_auc,
        "brier": brier,
        "c@1": c_at_1,
        "f1": f1,
        "f05u": f05u,
        "mean": mean_score,
        "tn": int(cm[0][0]),
        "fp": int(cm[0][1]),
        "fn": int(cm[1][0]),
        "tp": int(cm[1][1]),
    }

In [11]:
#Training arguments
training_args = TrainingArguments(
    output_dir="/home/tawe7542/Roberta_base/results",
    logging_dir="/home/tawe7542/Roberta_base/logs",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=4,
    load_best_model_at_end=True,
    metric_for_best_model="mean",
    greater_is_better=True,

    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [12]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_pan_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
#Train
trainer.train()

/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Roc Auc,Brier,C@1,F1,F05u,Mean,Tn,Fp,Fn,Tp
1,0.032739,0.041131,0.999149,0.993171,0.993034,0.994585,0.995491,0.995086,1268,9,16,2296


/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
trainer.train(resume_from_checkpoint="/home/tawe7542/Roberta_base/results/checkpoint-8892")

/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Roc Auc,Brier,C@1,F1,F05u,Mean,Tn,Fp,Fn,Tp
2,0.081500,0.354292,0.999487,0.947008,0.943996,0.958307,0.935374,0.956834,1078,199,2,2310
3,0.004226,0.307468,0.998653,0.962341,0.960992,0.970576,0.954448,0.969402,1140,137,3,2309


/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=8892, training_loss=0.01637004617464637, metrics={'train_runtime': 113144.4668, 'train_samples_per_second': 0.838, 'train_steps_per_second': 0.105, 'total_flos': 0.0, 'train_loss': 0.01637004617464637, 'epoch': 3.0})

In [13]:
trainer.train(resume_from_checkpoint="/home/tawe7542/Roberta_base/results/checkpoint-8892")

/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Roc Auc,Brier,C@1,F1,F05u,Mean,Tn,Fp,Fn,Tp
4,0.000056,0.101654,0.998969,0.988750,0.988576,0.991196,0.987000,0.990898,1240,37,4,2308


TrainOutput(global_step=11856, training_loss=0.0011342278076401577, metrics={'train_runtime': 38043.424, 'train_samples_per_second': 2.493, 'train_steps_per_second': 0.312, 'total_flos': 0.0, 'train_loss': 0.0011342278076401577, 'epoch': 4.0})

In [17]:
#Evaluation on validation dataset
pred_output = trainer.predict(val_dataset)

print(pred_output.metrics)        
logits = pred_output.predictions
labels = pred_output.label_ids

/home/tawe7542/Roberta_base/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.04113052785396576, 'test_roc_auc': 0.9991486656388107, 'test_brier': 0.9931707917712629, 'test_c@1': 0.9930342713847868, 'test_f1': 0.9945852285033572, 'test_f05u': 0.9954908081859176, 'test_mean': 0.995085953096827, 'test_tn': 1268, 'test_fp': 9, 'test_fn': 16, 'test_tp': 2296, 'test_runtime': 1079.3812, 'test_samples_per_second': 3.325, 'test_steps_per_second': 0.416}


In [15]:
#model saving
trainer.save_model("/home/tawe7542/Roberta_base/final_fusion_model")
tokenizer.save_pretrained("/home/tawe7542/Roberta_base/final_fusion_model")

('/home/tawe7542/Roberta_base/final_fusion_model/tokenizer_config.json',
 '/home/tawe7542/Roberta_base/final_fusion_model/tokenizer.json')